# 04. Эксперименты CP2

Градиентный бустинг (**LightGBM** при наличии; иначе `HistGradientBoostingRegressor` из sklearn), **XGBoost** при наличии; иначе `GradientBoostingRegressor`, ансамбль **Stacking**, **PCA**.

На Linux / в Docker библиотеки `lightgbm` и `xgboost` ставятся из `requirements.txt`. На macOS без OpenMP LightGBM может не импортироваться — ноутбук автоматически переключается на sklearn.

**Протокол:** time-based split, подбор гиперпараметров только на train (CV), отчёт по val/test.

In [1]:
import sys
from pathlib import Path

ROOT = Path().resolve().parent if Path().resolve().name == "notebooks" else Path().resolve()
sys.path.insert(0, str(ROOT))

import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import loguniform, randint as sp_randint, uniform
from sklearn.base import clone
from sklearn.decomposition import PCA
from sklearn.ensemble import (
    GradientBoostingRegressor,
    HistGradientBoostingRegressor,
    RandomForestRegressor,
    StackingRegressor,
)
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer

from src.config import MODELS_DIR, REPORT_IMAGES_DIR, SEED
from src.data_loader import load_raw
from src.preprocessing import (
    build_full_pipeline,
    build_preprocessor,
    prepare_features,
    split_xy,
    time_or_random_split,
)
from src.utils import regression_metrics, set_seed

try:
    from lightgbm import LGBMRegressor

    HAS_LGBM = True
except Exception:
    HAS_LGBM = False
    LGBMRegressor = None  # type: ignore[misc, assignment]

try:
    from xgboost import XGBRegressor

    HAS_XGB = True
except Exception:
    HAS_XGB = False
    XGBRegressor = None  # type: ignore[misc, assignment]

set_seed(SEED)
REPORT_IMAGES_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
print("boost libraries:", {"lightgbm": HAS_LGBM, "xgboost": HAS_XGB})


boost libraries: {'lightgbm': False, 'xgboost': False}


## Данные и сплит

In [2]:
df = load_raw()
train_df, val_df, test_df = time_or_random_split(df)
cv = KFold(n_splits=5, shuffle=True, random_state=SEED)
print(f"train={len(train_df)}, val={len(val_df)}, test={len(test_df)}")

experiment_rows: list[dict] = []

train=7000, val=1500, test=1500


## Таблица экспериментов

In [3]:
def add_row(name: str, eid: str, key_params: str, pipe, notes: str = "") -> None:
    for split_name, split_df in (("val", val_df), ("test", test_df)):
        X, y = split_xy(split_df)
        pred = pipe.predict(X)
        m = regression_metrics(np.asarray(y), np.asarray(pred))
        experiment_rows.append(
            {
                "experiment_id": eid,
                "model": name,
                "key_params": key_params,
                "split": split_name,
                "RMSE": m["RMSE"],
                "MAE": m["MAE"],
                "R2": m["R2"],
                "notes": notes,
            }
        )

## Эксперимент 1 — RandomForest

In [4]:
rf_est = RandomForestRegressor(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=1,
    n_jobs=-1,
    random_state=SEED,
)
pipe_rf = build_full_pipeline(rf_est)
X_tr, y_tr = split_xy(train_df)
pipe_rf.fit(X_tr, y_tr)
add_row(
    "RandomForest",
    "exp01_rf",
    "n_estimators=300,max_depth=None,min_samples_leaf=1",
    pipe_rf,
)

## Эксперимент 2 — градиентный бустинг + RandomizedSearchCV

In [5]:
if HAS_LGBM:
    boost_label = "LightGBM"
    boost_base = LGBMRegressor(
        objective="regression",
        random_state=SEED,
        verbosity=-1,
        n_estimators=200,
    )
    param_boost = {
        "model__learning_rate": loguniform(0.02, 0.2),
        "model__n_estimators": sp_randint(150, 600),
        "model__num_leaves": sp_randint(20, 100),
        "model__max_depth": sp_randint(4, 16),
        "model__subsample": uniform(0.6, 0.35),
        "model__colsample_bytree": uniform(0.6, 0.35),
    }
else:
    boost_label = "HistGradientBoosting(sklearn)"
    boost_base = HistGradientBoostingRegressor(random_state=SEED)
    param_boost = {
        "model__learning_rate": loguniform(0.02, 0.2),
        "model__max_iter": sp_randint(150, 600),
        "model__max_depth": sp_randint(4, 16),
        "model__min_samples_leaf": sp_randint(10, 80),
        "model__l2_regularization": uniform(0.0, 2.0),
    }

pipe_lgb = build_full_pipeline(boost_base)
search_lgb = RandomizedSearchCV(
    pipe_lgb,
    param_boost,
    n_iter=20,
    cv=cv,
    scoring="neg_root_mean_squared_error",
    refit=True,
    random_state=SEED,
    n_jobs=-1,
    verbose=1,
)
search_lgb.fit(X_tr, y_tr)
best_lgb = search_lgb.best_estimator_
print("best", boost_label, search_lgb.best_params_)
add_row(
    boost_label,
    "exp02_boost_tuned",
    json.dumps(search_lgb.best_params_, sort_keys=True, default=str),
    best_lgb,
)


Fitting 5 folds for each of 20 candidates, totalling 100 fits


best HistGradientBoosting(sklearn) {'model__l2_regularization': np.float64(1.3684660530243138), 'model__learning_rate': np.float64(0.05510391929902154), 'model__max_depth': 10, 'model__max_iter': 577, 'model__min_samples_leaf': 17}


## Эксперимент 3 — XGBoost или sklearn GradientBoosting

In [6]:
if HAS_XGB:
    tree_label = "XGBoost"
    tree_base = XGBRegressor(
        objective="reg:squarederror",
        random_state=SEED,
        n_estimators=300,
        tree_method="hist",
        n_jobs=-1,
    )
    param_tree = {
        "model__learning_rate": loguniform(0.02, 0.2),
        "model__max_depth": sp_randint(3, 12),
        "model__subsample": uniform(0.6, 0.35),
        "model__colsample_bytree": uniform(0.6, 0.35),
        "model__min_child_weight": sp_randint(1, 10),
    }
else:
    tree_label = "GradientBoosting(sklearn)"
    tree_base = GradientBoostingRegressor(random_state=SEED)
    param_tree = {
        "model__learning_rate": loguniform(0.02, 0.2),
        "model__n_estimators": sp_randint(100, 400),
        "model__max_depth": sp_randint(3, 10),
        "model__subsample": uniform(0.6, 0.35),
    }

pipe_xgb = build_full_pipeline(tree_base)
search_xgb = RandomizedSearchCV(
    pipe_xgb,
    param_tree,
    n_iter=15,
    cv=cv,
    scoring="neg_root_mean_squared_error",
    refit=True,
    random_state=SEED,
    n_jobs=-1,
    verbose=1,
)
search_xgb.fit(X_tr, y_tr)
best_xgb = search_xgb.best_estimator_
print("best", tree_label, search_xgb.best_params_)
add_row(
    tree_label,
    "exp03_tree_tuned",
    json.dumps(search_xgb.best_params_, sort_keys=True, default=str),
    best_xgb,
)


Fitting 5 folds for each of 15 candidates, totalling 75 fits


best GradientBoosting(sklearn) {'model__learning_rate': np.float64(0.0828917875602605), 'model__max_depth': 4, 'model__n_estimators': 121, 'model__subsample': np.float64(0.6024732068269011)}


## Эксперимент 4 — Stacking (RF + лучший бустинг → Ridge)

**Гипотеза:** мета-модель на выходах базовых регрессоров снизит ошибку относительно одиночных моделей.


In [7]:
estimators = [
    (
        "rf",
        build_full_pipeline(
            RandomForestRegressor(
                n_estimators=250,
                random_state=SEED,
                n_jobs=-1,
            )
        ),
    ),
    ("boost", best_lgb),
]
stack = StackingRegressor(
    estimators=estimators,
    final_estimator=Ridge(alpha=1.0, random_state=SEED),
    passthrough=False,
    cv=KFold(n_splits=5, shuffle=True, random_state=SEED),
    n_jobs=-1,
)
X_tr, y_tr = split_xy(train_df)
stack.fit(X_tr, y_tr)
add_row(
    "Stacking(RF+Boost->Ridge)",
    "exp04_stack",
    "final=Ridge(alpha=1),cv=5",
    stack,
)


## Эксперимент 5 — PCA (95% дисперсии) + тот же семейство бустинга

Визуализация: scree / кумулятивная объяснённая дисперсия на train после препроцессора.


In [8]:
prep_only = Pipeline(
    steps=[
        ("prep", FunctionTransformer(prepare_features, validate=False)),
        ("features", build_preprocessor()),
    ]
)
Xt_train = prep_only.fit_transform(X_tr, y_tr)
n_feat = Xt_train.shape[1]
n_comp_scree = min(60, n_feat)
pca_scree = PCA(n_components=n_comp_scree, random_state=SEED)
pca_scree.fit(Xt_train)
cumvar = np.cumsum(pca_scree.explained_variance_ratio_)
plt.figure(figsize=(6, 4))
plt.plot(np.arange(1, len(cumvar) + 1), cumvar, marker=".")
plt.xlabel("n_components")
plt.ylabel("cumulative explained variance")
plt.title("PCA cumulative variance (train)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(REPORT_IMAGES_DIR / "pca_scree_cp2.png", dpi=150)
plt.close()

pca_inner = clone(best_lgb.named_steps["model"])
pipe_pca_lgb = Pipeline(
    steps=[
        ("prep", FunctionTransformer(prepare_features, validate=False)),
        ("features", build_preprocessor()),
        ("pca", PCA(n_components=0.95, random_state=SEED)),
        ("model", pca_inner),
    ]
)
pipe_pca_lgb.fit(X_tr, y_tr)
add_row(
    f"{boost_label}+PCA(0.95)",
    "exp05_pca_boost",
    "pca=n_components=0.95,clone(best_lgb.model)",
    pipe_pca_lgb,
)

pca2 = Pipeline(
    steps=[
        ("prep", FunctionTransformer(prepare_features, validate=False)),
        ("features", build_preprocessor()),
        ("pca", PCA(n_components=2, random_state=SEED)),
    ]
)
Z = pca2.fit_transform(X_tr, y_tr)
plt.figure(figsize=(6, 5))
plt.scatter(Z[:, 0], Z[:, 1], c=y_tr, cmap="viridis", s=8, alpha=0.6)
plt.colorbar(label="profit")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA-2 projection colored by profit (train)")
plt.tight_layout()
plt.savefig(REPORT_IMAGES_DIR / "pca2_scatter_cp2.png", dpi=150)
plt.close()


## Сводка и финальная модель (выбор по val RMSE)

In [9]:
exp_df = pd.DataFrame(experiment_rows)
exp_path = REPORT_IMAGES_DIR / "cp2_experiments.csv"
exp_df.sort_values(["experiment_id", "split"]).to_csv(exp_path, index=False)
print("Saved", exp_path)

val_df_metrics = exp_df[exp_df["split"] == "val"].copy()
best_row = val_df_metrics.loc[val_df_metrics["RMSE"].idxmin()]
print("Best by val RMSE:", best_row[["experiment_id", "model", "RMSE"]])
best_eid = str(best_row["experiment_id"])

fitted_by_id = {
    "exp01_rf": pipe_rf,
    "exp02_boost_tuned": best_lgb,
    "exp03_tree_tuned": best_xgb,
    "exp04_stack": stack,
    "exp05_pca_boost": pipe_pca_lgb,
}
final_pipe = fitted_by_id[best_eid]

X_va, y_va = split_xy(val_df)
X_te, y_te = split_xy(test_df)
print("Final val", regression_metrics(np.asarray(y_va), final_pipe.predict(X_va)))
print("Final test", regression_metrics(np.asarray(y_te), final_pipe.predict(X_te)))

Saved /Users/stepannovichihin/Downloads/GitHub/Учеба/hseml-group-project-novichihin-1/report/images/cp2_experiments.csv
Best by val RMSE: experiment_id             exp03_tree_tuned
model            GradientBoosting(sklearn)
RMSE                          26777.738455
Name: 4, dtype: object
Final val {'RMSE': 26777.738455196468, 'MAE': 7697.553967944279, 'R2': 0.8631042825124062}
Final test {'RMSE': 64019.33219113364, 'MAE': 14924.581557888056, 'R2': 0.7334762990071791}


In [10]:
from src.data_shift import save_shift_report

save_shift_report(train_df, test_df, REPORT_IMAGES_DIR)
print("Saved train_test_shift_cp2.csv / .png")


Saved train_test_shift_cp2.csv / .png


## Ablation: time-based split vs случайный split

Тот же тип модели, что и финальный пайплайн (`clone`), обучение на random 70/15/15 без учёта времени (используется `time_or_random_split(..., time_order=False)` (те же строки и колонки, включая `start_date`)).


In [11]:
from sklearn.base import clone

train_r, val_r, test_r = time_or_random_split(df, time_order=False)
pipe_rand = clone(final_pipe)
Xtr_r, ytr_r = split_xy(train_r)
pipe_rand.fit(Xtr_r, ytr_r)

ab_rows = []
for split_name, pipe_use, sval, stest in (
    ("time", final_pipe, val_df, test_df),
    ("random", pipe_rand, val_r, test_r),
):
    Xv, yv = split_xy(sval)
    Xt, yt = split_xy(stest)
    pv = pipe_use.predict(Xv)
    pt = pipe_use.predict(Xt)
    mv = regression_metrics(np.asarray(yv), pv)
    mt = regression_metrics(np.asarray(yt), pt)
    ab_rows.append(
        {
            "split_strategy": split_name,
            "RMSE_val": mv["RMSE"],
            "RMSE_test": mt["RMSE"],
            "R2_val": mv["R2"],
            "R2_test": mt["R2"],
        }
    )

ab_df = pd.DataFrame(ab_rows)
ab_df.to_csv(REPORT_IMAGES_DIR / "split_ablation_cp2.csv", index=False)
print(ab_df)


  split_strategy      RMSE_val     RMSE_test    R2_val   R2_test
0           time  26777.738455  64019.332191  0.863104  0.733476
1         random  34324.933110  28767.205918  0.840633  0.771101


## PCA sweep: RMSE (val) vs число компонент

Фиксированный `GradientBoostingRegressor` после PCA на препроцессоре.


In [12]:
from sklearn.ensemble import GradientBoostingRegressor

X_va, y_va = split_xy(val_df)
sweep = []
for ncomp in [20, 40, 60]:
    pipe_s = Pipeline(
        steps=[
            ("prep", FunctionTransformer(prepare_features, validate=False)),
            ("features", build_preprocessor()),
            ("pca", PCA(n_components=ncomp, random_state=SEED)),
            (
                "model",
                GradientBoostingRegressor(
                    random_state=SEED, max_depth=4, learning_rate=0.08, n_estimators=200
                ),
            ),
        ]
    )
    pipe_s.fit(X_tr, y_tr)
    pred = pipe_s.predict(X_va)
    m = regression_metrics(np.asarray(y_va), pred)
    sweep.append({"n_components": ncomp, "RMSE_val": m["RMSE"], "R2_val": m["R2"]})

pipe_s95 = Pipeline(
    steps=[
        ("prep", FunctionTransformer(prepare_features, validate=False)),
        ("features", build_preprocessor()),
        ("pca", PCA(n_components=0.95, random_state=SEED)),
        (
            "model",
            GradientBoostingRegressor(
                random_state=SEED, max_depth=4, learning_rate=0.08, n_estimators=200
            ),
        ),
    ]
)
pipe_s95.fit(X_tr, y_tr)
m95 = regression_metrics(np.asarray(y_va), pipe_s95.predict(X_va))
sweep.append({"n_components": "0.95_var", "RMSE_val": m95["RMSE"], "R2_val": m95["R2"]})

pd.DataFrame(sweep).to_csv(REPORT_IMAGES_DIR / "pca_sweep_cp2.csv", index=False)
print(pd.DataFrame(sweep))


  n_components      RMSE_val    R2_val
0           20  35024.811955  0.765796
1           40  37123.535456  0.736888
2           60  29764.352623  0.830864
3     0.95_var  29804.809932  0.830404


## Permutation importance (validation)

По финальному пайплайну на `val`.


In [13]:
from src.interpretability import save_permutation_importance

save_permutation_importance(final_pipe, X_va, y_va, REPORT_IMAGES_DIR)
print("Saved permutation_importance_cp2.csv / .png")


Saved permutation_importance_cp2.csv / .png


In [14]:
import joblib

final_path = MODELS_DIR / "final_cp2.joblib"
joblib.dump({"pipeline": final_pipe, "best_experiment_id": best_eid}, final_path)
print("Saved", final_path)

Saved /Users/stepannovichihin/Downloads/GitHub/Учеба/hseml-group-project-novichihin-1/models/final_cp2.joblib
